# Evaluate product-return evidence with the Decisions API

A customer reports a damaged item. Before a support agent handles the return,
the application needs a usable product photo, order information, and a clear
understanding of the complaint. This example checks that evidence and prepares
an appropriate next step for a person.

The [Decisions API](https://developers.openai.com/api/docs/guides/decisions)
returns typed answers about text, images, and JSON. You will send all three with
`gpt-6-luna` to `POST /v1/decisions`, asking one `choice` question and two
independent `predicate` questions in each request.

Select **Run All** to make **ten billable API requests**. You will see the photos,
actual API responses, local handoff records, and an evaluation table. The helper
uses Python's standard library. Use Python 3.10+ with a working Jupyter kernel
or VS Code notebook support. See the [run guide](README.md)
for a Terminal alternative that saves a report with the same images and results.

![A synthetic blue metal bottle with a large dent](images/dented_bottle.png)

All cases are fictional and all photos are synthetic illustrations. They are
bundled locally; no dataset download is needed. This application creates local
recommendations. A person must approve any return; no tickets or refunds are sent.

## 1. Load the cases

Start the kernel in the repository root or `examples/decisions/product_returns/`. The final two
cases are snapshots of one conversation: an unclear photo, then a clearer photo.
The reference outcomes live in a separate file and are never sent to the API.

In [ ]:
import getpass
import json
import os
from pathlib import Path
import sys
import warnings

from IPython.display import Image, Markdown, display

# Support a kernel started in the repository root or this use case folder.
example_dir = next((path for path in (Path.cwd(), Path.cwd() / "examples" / "decisions" / "product_returns")
                    if (path / "return_evidence.py").is_file()), None)
if example_dir is None:
    raise RuntimeError("Start the kernel in the repository root or examples/decisions/product_returns")
sys.path.insert(0, str(example_dir))

from return_evidence import (API_URL, QUESTIONS, build_request, grade_result,
                             load_cases, load_expectations, parse_answers, recommend_next_step,
                             request_decision)
from return_reporting import display_request, write_report

# Load evidence and local reference outcomes separately.
cases = load_cases(example_dir)
expectations = load_expectations(cases, example_dir)
report_path = Path.cwd() / "outputs" / "product-return-results.html"
# Check that all images are available before making paid requests.
payloads = [build_request(case, example_dir) for case in cases]
print(f"Loaded {len(cases)} cases. A full run makes {len(cases)} billable requests.")

## 2. Ask three independent questions

- `reported_issue` uses `choice` to classify the customer's stated problem.
- `photo_usable` uses `predicate` to assess whether the latest photo shows the product clearly.
- `visible_damage` uses `predicate` to assess visible product damage, excluding packaging damage.

The two image questions assess the photo directly. Neither uses another answer
as its input. For the final conversation, their instructions focus on the latest
photo. The issue question uses the conversation to interpret the latest request.

A predicate returns a `probability`, while a choice returns its selected `choice`
and a separate `confidence`. These are model estimates, not guarantees.

In [ ]:
# Inspect the three questions that share each case's evidence.
print(json.dumps(QUESTIONS, indent=2))

## 3. Inspect the request

`build_request` combines user text, an `input_image` part containing a bundled
photo, and an `input_json` part containing the order. The final case also includes
prior user and assistant messages. The question instructions tell the model to treat customer claims as evidence.

The display below omits image bytes for readability. The actual request sends
the complete base64 image. The Authorization header is supplied separately and
is never printed.

In [ ]:
# Preview one request without printing the full base64 photo.
payload = build_request(cases[0], example_dir)
print("POST", API_URL)
print(json.dumps(display_request(payload), indent=2))

## 4. Apply an illustrative handoff policy

The API assesses evidence. Application code decides the next step:

| Condition | Next step | Responsible queue |
| --- | --- | --- |
| Missing order details or an unusable product photo | Request information | Customer support |
| Clear product damage within the 30-day demo return window | Prepare for review | Returns agent |
| Conflicting or uncertain evidence, another return reason, or a policy exception | Specialist review | Returns specialist |
| Refused, missing, duplicate, or invalid API answers | Specialist review | Returns specialist |

`recommend_next_step` checks these conditions in a fixed order: unusable API
answers, order completeness, return window, reported issue, photo usability,
then visible damage. Earlier checks take precedence. For example, an outside-window
case goes to a specialist even if its photo is unclear.

The illustrative thresholds are 0.8 for strong evidence and 0.2 for little evidence
of damage. Values between them go to a specialist when the photo is usable.
These cutoffs have not been calibrated. Test thresholds on representative development
cases and evaluate them on a separate held-out set before relying on them.
The date and window checks use code, not model judgment.

Each local handoff record identifies an owner queue, a reason code, and the
policy version. It is a proposed assignment, not proof that a person received,
reviewed, or approved the case. A real helpdesk integration must enforce those steps.

## 5. Run ten cases and inspect the results

This cell uses `OPENAI_API_KEY` or requests a key with hidden input. It makes one
real request per case, displays the images, request JSON with image bytes omitted,
and returned answers, and saves local HTML and JSON reports after each completed case. API requests have a
30-second timeout, reject redirects, and do not retry automatically. Network or
HTTP failures stop the run; inspect the error and any saved results before rerunning.

Running this cell again makes another ten paid requests. Check
[current pricing](https://developers.openai.com/api/docs/guides/decisions#pricing-and-availability).
Keep keys out of code cells and saved outputs.

In [ ]:
rows = []
api_key = None
try:
    # Use an environment key or a hidden prompt; never put the key in a cell.
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            api_key = getpass.getpass("OpenAI API key (hidden): ")
    if not api_key:
        raise RuntimeError("No key supplied; no API requests made")
    # Reset the report for this run, then save each completed case.
    write_report([], report_path, example_dir)
    for index, (case, payload) in enumerate(zip(cases, payloads), 1):
        print(f"\nExample {index}/{len(cases)}: {case['title']}")
        # Show the conversation and photos before inspecting the API answer.
        for turn in case["conversation"]:
            print(f"{turn['role']}: {turn['text']}")
            if turn.get("image"):
                display(Image(filename=str(example_dir / "images" / turn["image"]), width=380))
        print("Order:", json.dumps(case["order"]))
        # Display the same request being sent, with only image bytes omitted.
        print("API REQUEST: POST", API_URL)
        print(json.dumps(display_request(payload), indent=2))
        # Send one real request containing all three independent questions.
        response, latency_ms = request_decision(payload, api_key)
        print("ACTUAL API RESPONSE:", json.dumps(response, indent=2))
        # Apply local policy checks to the validated API answers.
        handoff = recommend_next_step(case, parse_answers(response))
        print("APPLICATION HANDOFF:", json.dumps(handoff, indent=2))
        # Grade the next step locally and retain the actual API response.
        rows.append({"case": case, "request": display_request(payload),
                     "response": response, "handoff": handoff,
                     "grade": grade_result(case["id"], handoff, expectations),
                     "latency_ms": latency_ms})
        write_report(rows, report_path, example_dir)
finally:
    # Drop this local key reference even if a request fails.
    api_key = None
print("Saved report:", report_path.resolve())

## 6. Evaluate the application outcomes

The following checks compare the same live results with the independently authored
reference outcomes. They make no additional API calls and use no model judge.
A missed escalation means a specialist case was sent elsewhere; an unnecessary
escalation means another expected outcome was sent to a specialist.

These ten examples are a development demonstration, not a held-out benchmark.
Photos are reused, and cases 9 and 10 belong to the same conversation. Matching
these labels does not establish production accuracy or calibrated confidence.
Inspect every failure before changing prompts, thresholds, or labels.
An unusable API answer can correctly trigger specialist review and match the
reference outcome. The summary reports these fallbacks separately from outcome
matches so they are visible.


In [ ]:
# Compare the existing results with reference labels; no new API calls.
table = ["| Case | Expected | Actual | Match | Latency ms |",
         "| --- | --- | --- | --- | --- |"]
for row in rows:
    grade = row["grade"]
    table.append(f"| {row['case']['id']} | {grade['expected']} | {grade['actual']} | "
                 f"{'PASS' if grade['passed'] else 'FAIL'} | {row['latency_ms']} |")
display(Markdown("\n".join(table)))
print(f"Completed API requests: {len(rows)}/{len(cases)}")
print("Expected outcomes matched:", sum(r["grade"]["passed"] for r in rows))
print("Unusable API answers:", sum(parse_answers(r["response"]) is None for r in rows))
print("Missed escalations:", sum(r["grade"]["missed_escalation"] for r in rows))
print("Unnecessary escalations:", sum(r["grade"]["unnecessary_escalation"] for r in rows))
print("Total tokens:", sum(r["response"].get("usage", {}).get("total_tokens", 0) for r in rows))

## 7. Compare the conversation before and after clarification

Cases 9 and 10 share an order. The first photo is too blurred to inspect. In the
second snapshot, the customer supplies a clear photo after an assistant asks for
one. Compare the recommendations below. These are two explicit API requests;
Decisions does not retain the conversation between them.

In [ ]:
# Compare the two saved snapshots before and after a clearer photo.
for row in rows[-2:]:
    print(row["case"]["title"], "->", row["handoff"]["next_step"], row["handoff"]["reason"])

## Adapt the workflow

Replace the fictional inputs with data and photos you are authorized to use.
Keep an independently labeled development set and a held-out regression set;
keep related conversations and reused images in the same split. Measure behavior
on missing evidence, conflicting claims, and policy exceptions as well as clear cases.

Record human overrides and verified outcomes separately from model answers. An
owner queue in a JSON record is not an authenticated approval. If you connect a
helpdesk or refund tool, implement authorization, durable status tracking, and
outcome verification in that application.

Run the [offline regression tests](README.md#developer-checks)
when changing the parser, transport, or policy. The [image provenance file](images/provenance.json)
records the generation prompts and checksums. All seven images were visually
inspected before assigning the reference outcomes.